# LT separation: data-only bin selection

Compare one or several epsilon settings in six linked views. The editor uses exact axis coordinates, persistent numbered corners, numeric editing, and undo history. The same Matplotlib figure exports to PDF/SVG without notebook controls.

All edges use **data only**. Phi bins remain uniform. The later SIMC section computes representative coordinates for the reference setting; it does not choose edges or calculate cross sections.


## Running and editing

Use `/group/nps/singhav/software/python/bin/python`. From the repository, launch `PATH=/group/nps/singhav/software/python/bin:$PATH /group/nps/singhav/software/python/bin/python -m jupyterlab src/xsec_extract/xsec_bin_helper.ipynb`, select Python 3, and run all cells. The first cell checks the interpreter. `ipywidgets` and `anywidget` must be enabled in the frontend; no Plotly extension is needed.

1. Click four distinct convex corners on either upper map. Each click is acknowledged with coordinates, and every number remains on **both** plots. Wait for the acknowledgement before the next click.
2. To move a corner, select its number then click its new position, or edit xB and Q2 and press **Set corner**. After closure the target remains corner 4. **Undo** restores previous positions and can undo **Clear**.
3. Edit outer edges and press **Apply outer limits**. Modes and bin counts apply immediately. Manual arrays live in the expandable panel; xB requires one array per Q2 bin.
4. Inspect selected counts, the per-setting sparse-bin table, and t-prime/phi slices. **Map min** controls coverage coloring; **Sparse <** flags the proposed analysis bins. These are separate thresholds.
5. Rerun downstream review and representative-coordinate cells after changing the cut. Incomplete, invalid, and stale selections block downstream use/export. Copy configuration to preserve your choices across a kernel restart. Static figure export needs no SIMC; the scientific JSON export retains its stricter safeguards.

For a single setting, set `KINEMATICS = ["KinC_x36_4"]` and `REFERENCE_KINEMATIC = "KinC_x36_4"` in the configuration, or launch with `NPS_BIN_SETTINGS=KinC_x36_4`. See `README_xsec_bin_helper.md` for automated validation.


In [10]:
from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.path import Path as PolygonPath
import uproot
from IPython.display import Markdown, display
import os
import sys
if Path(sys.executable).resolve() != Path("/group/nps/singhav/software/python/bin/python").resolve():
    raise RuntimeError(f"Wrong kernel: {sys.executable}; select the Hall C/NPS Python kernel")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.titleweight": "bold"})
pd.set_option("display.max_colwidth", 140)

## Configuration

`KINEMATICS` accepts one or several keys from `config/nps_simulation_kinematics*.csv`. The 407 cm row is selected by its own key, `KinC_x36_5_407`. The CSV supplies the beam energy used for the epsilon diagnostic. `REFERENCE_KINEMATIC` selects the setting for the existing SIMC representative-coordinate section.

Use `Q2_LIMITS`, `XB_LIMITS`, and `TPRIME_LIMITS` as outer edges. Optional `DIAMOND_XB_Q2_VERTICES` restricts the common phase space. Candidate modes are `uniform`, `count_quantile`, and `yield_quantile`; `MANUAL_TPRIME_EDGES` and `MANUAL_Q2_EDGES` supply manual arrays; `XB_EDGES_BY_Q2` supplies manual arrays within each Q2 bin. `XB_EDGES_BY_Q2` can override the global xB mode with a separate manual array for every Q2 bin. These are selection aids; review coverage for **each** setting before freezing edges. `W_Q2_MAP_BINS` sets the W/Q2 display grid independently of `OVERLAP_BINS` for xB/Q2; `MIN_EVENTS_PER_CELL` sets the initial **Map min** display threshold.

In [11]:
REPO_ROOT = Path("/w/hallc-scshelf2102/nps/singhav/nps_analysis/pi0_analysis/root_analysis_env_main")
DATA_FILES = {
    "KinC_x36_4": REPO_ROOT / "output/KinC_x36_4/root/combined_branches_LH2.root",
    "KinC_x36_5_407": REPO_ROOT / "output/KinC_x36_5_407/KinC_x36_5/root/combined_branches_LH2.root",
}
KINEMATICS = ["KinC_x36_4", "KinC_x36_5_407"]  # Or ["KinC_x36_4"]
REFERENCE_KINEMATIC = "KinC_x36_5_407"
SIM_FILES = {
    "KinC_x36_5_407": Path("/volatile/hallc/nps/singhav/nps_smearing/smear_x36_5_407/smearing_output/KinC_x36_5_407/root/simc_pi0_analysis_output_smeared.root"),
}
SIM_FILE = SIM_FILES.get(REFERENCE_KINEMATIC)
DATA_TREE = "physics"
SIM_TREE = "simulation"

N_TPRIME = 5
N_PHI = 15
TPRIME_LIMITS = (-0.7, 0.0)     # GeV^2
Q2_LIMITS = (3.0, 5.0)           # GeV^2
XB_LIMITS = (0.25, 0.455)
MMISS_LIMITS = (0.80, 1.10)      # GeV
DIAMOND_XB_Q2_VERTICES = None   # Example: [(0.30, 3.2), (0.39, 4.4), (0.45, 4.8), (0.34, 3.6)]

TPRIME_EDGE_MODE = "count_quantile"
MANUAL_TPRIME_EDGES = None
Q2_EDGE_MODE = "count_quantile"
N_Q2 = 2
MANUAL_Q2_EDGES = None
XB_EDGE_MODE = "count_quantile"
N_XB_PER_Q2 = 2
XB_EDGES_BY_Q2 = None
PHI_EDGES = np.linspace(0.0, 2.0 * np.pi, N_PHI + 1)

# Coverage colors use every event; selected counts and bin statistics are exact.
OVERLAP_BINS = (60, 50)       # xB/Q2 coverage map
W_Q2_MAP_BINS = (120, 100)     # finer W/Q2 coverage map
MIN_EVENTS_PER_CELL = 1       # keep the smaller W/Q2 cells visible
MAX_SCATTER_PER_SETTING = 12000

phi_cpp_edges = ["0.0"] + [f"{2 * i}*TMath::Pi()/{N_PHI}" for i in range(1, N_PHI)] + ["2*TMath::Pi()"]
print("std::vector<double> phi_bin_edges = {")
phi_cpp_rows = [phi_cpp_edges[:4]] + [phi_cpp_edges[start:start + 3] for start in range(4, len(phi_cpp_edges), 3)]
for index, row in enumerate(phi_cpp_rows):
    suffix = "," if index + 1 < len(phi_cpp_rows) else ""
    print("    " + ", ".join(row) + suffix)
print("};")

PROTON_MASS_GEV = 0.9382720813
WRITE_EXPORT = False
EXPORT_PATH = REPO_ROOT / "src/xsec_extract/xsec_bin_selection.json"

# Optional launch override for reproducible single/multi-setting runs.
if os.environ.get("NPS_BIN_SETTINGS"):
    KINEMATICS = [v.strip() for v in os.environ["NPS_BIN_SETTINGS"].split(",")]
    if REFERENCE_KINEMATIC not in KINEMATICS:
        REFERENCE_KINEMATIC = KINEMATICS[0]
    SIM_FILE = SIM_FILES.get(REFERENCE_KINEMATIC)

if not KINEMATICS or len(KINEMATICS) != len(set(KINEMATICS)):
    raise ValueError("KINEMATICS must contain distinct settings")
if REFERENCE_KINEMATIC not in KINEMATICS:
    raise ValueError("REFERENCE_KINEMATIC must be one of KINEMATICS")
if not all(lo < hi for lo, hi in (TPRIME_LIMITS, Q2_LIMITS, XB_LIMITS, MMISS_LIMITS)):
    raise ValueError("Every outer limit must increase")
if DIAMOND_XB_Q2_VERTICES is not None and len(DIAMOND_XB_Q2_VERTICES) != 4:
    raise ValueError("A diamond polygon needs exactly four (xB, Q2) vertices")

std::vector<double> phi_bin_edges = {
    0.0, 2*TMath::Pi()/15, 4*TMath::Pi()/15, 6*TMath::Pi()/15,
    8*TMath::Pi()/15, 10*TMath::Pi()/15, 12*TMath::Pi()/15,
    14*TMath::Pi()/15, 16*TMath::Pi()/15, 18*TMath::Pi()/15,
    20*TMath::Pi()/15, 22*TMath::Pi()/15, 24*TMath::Pi()/15,
    26*TMath::Pi()/15, 28*TMath::Pi()/15, 2*TMath::Pi()
};


## 1. Load and select data

Each file is opened independently. Finite physical events passing missing mass form the retained input sample, so outer t-prime limits can later expand. The maps show this coverage at the current t-prime limits. Q2, xB and the same xB/Q2 diamond then select both settings. Display weights retain the existing convention: `pi0_weight * scale * (run charge / setting total charge)`; count quantiles ignore weights. Yield quantiles use positive finite weights, as before.


In [12]:
DATA_BRANCHES = [
    "Q2", "xB", "W", "t", "tmin", "phi", "mmiss_all",
    "pi0_weight", "scale", "charge_uC", "run_number",
]

def open_checked_tree(path, tree_name, required_branches):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    root_file = uproot.open(path)
    if tree_name not in root_file:
        raise KeyError(f"Tree {tree_name!r} is absent from {path}")
    tree = root_file[tree_name]
    missing = sorted(set(required_branches) - set(tree.keys()))
    if missing:
        raise KeyError(f"Missing branches in {path}: {missing}")
    return root_file, tree

def tree_frame(tree, branches):
    arrays = tree.arrays(branches, library="np")
    return pd.DataFrame({name: arrays[name] for name in branches})

# Require exactly one matching simulation-kinematics row per setting.
kinematics_tables = []
for csv_path in sorted((REPO_ROOT / "config").glob("nps_simulation_kinematics*.csv")):
    table = pd.read_csv(csv_path, skipinitialspace=True)
    table.columns = table.columns.str.strip()
    table["kin_old"] = table["kin_old"].astype(str).str.strip()
    table["target"] = table["target"].astype(str).str.strip()
    table["csv_source"] = str(csv_path)
    kinematics_tables.append(table)
if not kinematics_tables:
    raise FileNotFoundError("No config/nps_simulation_kinematics*.csv files found")
kinematics_table = pd.concat(kinematics_tables, ignore_index=True)
settings_metadata = {}
for setting in KINEMATICS:
    if setting not in DATA_FILES:
        raise KeyError(f"Add a DATA_FILES path for {setting}")
    matches = kinematics_table.loc[(kinematics_table["kin_old"] == setting) & (kinematics_table["target"] == "LH2")]
    if len(matches) != 1:
        raise ValueError(f"Expected one LH2 kinematics CSV row for {setting}; found {len(matches)}")
    settings_metadata[setting] = matches.iloc[0]
display(pd.DataFrame(settings_metadata).T[["new_kin", "ebeam_gev", "hms_p_gev", "hms_theta_deg", "nps_theta_deg", "nps_target_distance_cm", "csv_source"]])

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.xsec_extract.xsec_bin_dashboard import ordered_polygon, polygon_mask

def diamond_mask(frame):
    return polygon_mask(frame, ordered_polygon(DIAMOND_XB_Q2_VERTICES))

all_data, coverage_frames, selected_frames, summary_rows = {}, [], [], []
for setting in KINEMATICS:
    root_file, tree = open_checked_tree(DATA_FILES[setting], DATA_TREE, DATA_BRANCHES)
    frame = tree_frame(tree, DATA_BRANCHES)
    root_file.close()
    frame["setting"] = setting
    frame["tprime"] = frame["t"] - frame["tmin"]
    frame["phi_wrapped"] = np.mod(frame["phi"], 2.0 * np.pi)
    valid_charge = np.isfinite(frame["charge_uC"]) & (frame["charge_uC"] > 0.0)
    run_charges = frame.loc[valid_charge, ["run_number", "charge_uC"]].drop_duplicates("run_number")
    total_charge_uC = run_charges["charge_uC"].sum()
    if not np.isfinite(total_charge_uC) or total_charge_uC <= 0.0:
        raise ValueError(f"{setting} has no valid positive total run charge")
    frame["data_weight"] = frame["pi0_weight"] * frame["scale"] * np.where(valid_charge, frame["charge_uC"] / total_charge_uC, 1.0)
    ebeam = float(settings_metadata[setting]["ebeam_gev"])
    nu = frame["Q2"] / (2.0 * PROTON_MASS_GEV * frame["xB"])
    eprime = ebeam - nu
    tan2 = frame["Q2"] / (4.0 * ebeam * eprime - frame["Q2"])
    frame["epsilon"] = np.where((eprime > 0) & (tan2 > 0), 1.0 / (1.0 + 2.0 * (1.0 + nu**2 / frame["Q2"]) * tan2), np.nan)
    finite = np.logical_and.reduce([np.isfinite(frame[name]) for name in ["Q2", "xB", "W", "tprime", "phi_wrapped", "mmiss_all", "data_weight"]])
    coverage = frame.loc[finite & frame["mmiss_all"].between(*MMISS_LIMITS, inclusive="both") & (frame["Q2"] > 0) & frame["xB"].between(1e-9, 1-1e-9)].copy()
    chosen = coverage.loc[coverage["tprime"].between(*TPRIME_LIMITS, inclusive="both") & coverage["Q2"].between(*Q2_LIMITS, inclusive="both") & coverage["xB"].between(*XB_LIMITS, inclusive="both") & diamond_mask(coverage)].copy()
    all_data[setting] = frame
    coverage_frames.append(coverage)
    selected_frames.append(chosen)
    summary_rows.append({"setting": setting, "all": len(frame), "coverage": len(coverage), "selected": len(chosen), "beam_GeV": ebeam, "median_epsilon_selected": chosen["epsilon"].median(), "charge_uC": total_charge_uC})

data = all_data[REFERENCE_KINEMATIC]
data_pre = pd.concat(coverage_frames, ignore_index=True)
data_selected = pd.concat(selected_frames, ignore_index=True)
if data_selected.empty:
    raise ValueError("No data remain after the chosen phase-space cuts")
display(pd.DataFrame(summary_rows))
print("All plots and bin candidates below use data; no simulation has been opened.")

,new_kin,ebeam_gev,hms_p_gev,hms_theta_deg,nps_theta_deg,nps_target_distance_cm,csv_source
KinC_x36_4,x36_q40_p4,8.455,2.562,24.78,10.1,357.0,/w/hallc-scshelf2102/nps/singhav/nps_analysis/pi0_analysis/root_analysis_env_main/config/nps_simulation_kinematics.csv
KinC_x36_5_407,x36_q40_p5,10.538,4.637,16.44,12.2,407.0,/w/hallc-scshelf2102/nps/singhav/nps_analysis/pi0_analysis/root_analysis_env_main/config/nps_simulation_kinematics_x36_5_407.csv


,setting,all,coverage,selected,beam_GeV,median_epsilon_selected,charge_uC
0,KinC_x36_4,76242,10803,9828,8.455,0.514739,880666.500
1,KinC_x36_5_407,108545,13676,11699,10.538,0.709886,382512.625


All plots and bin candidates below use data; no simulation has been opened.


## 2. Linked coverage and selection

Blue and orange identify exclusive coverage for the default pair; purple identifies cells occupied by **both** settings at `Map min`. For more settings, gray denotes a shared subset and purple denotes all. Full coverage remains visible when the diamond is edited. The black boundary intersected with dashed outer limits defines the selection; dotted lines mark bins. The legend includes each setting's selected count and median epsilon.

There are no floating hover boxes or trace-dependent click targets. The W projection samples the entire transformed boundary, and the ROOT W branch is checked numerically. The t-prime/phi map shows the least count among settings; crosses mark sparse cells. The expandable full-bin table identifies the sparse setting in every Q2/xB/t-prime/phi bin.


In [13]:
# The linked dashboard is constructed after the data-only t-prime quantile helper is defined.

## 3. Data-only $t'$ edge candidates

This table starts from the configuration cell. The dashboard below recomputes uniform, count-quantile, or data-yield-quantile edges whenever the cuts change. The saved edge arrays and event counts in later cells use the latest dashboard selection when those cells are run.

In [14]:
def weighted_quantile(values, probabilities, weights=None):
    values = np.asarray(values, dtype=float)
    probabilities = np.asarray(probabilities, dtype=float)
    if weights is None:
        return np.quantile(values[np.isfinite(values)], probabilities)
    weights = np.asarray(weights, dtype=float)
    keep = np.isfinite(values) & np.isfinite(weights) & (weights > 0.0)
    values, weights = values[keep], weights[keep]
    if values.size == 0:
        raise ValueError("No positive-weight finite values for weighted quantiles")
    order = np.argsort(values)
    values, weights = values[order], weights[order]
    cumulative = np.cumsum(weights) - 0.5 * weights
    cumulative /= weights.sum()
    return np.interp(probabilities, cumulative, values)

def quantile_edges(values, n_bins, limits, weights=None):
    lo, hi = map(float, limits)
    inside = np.isfinite(values) & (np.asarray(values) >= lo) & (np.asarray(values) <= hi)
    selected_values = np.asarray(values)[inside]
    selected_weights = None if weights is None else np.asarray(weights)[inside]
    interior = weighted_quantile(selected_values, np.arange(1, n_bins) / n_bins, selected_weights)
    edges = np.r_[lo, interior, hi]
    if np.any(np.diff(edges) <= 0.0):
        raise ValueError(f"Collapsed quantile edges: {edges}")
    return edges

tprime_candidates = {
    "uniform": np.linspace(*TPRIME_LIMITS, N_TPRIME + 1),
    "count_quantile": quantile_edges(data_selected["tprime"], N_TPRIME, TPRIME_LIMITS),
    "yield_quantile": quantile_edges(data_selected["tprime"], N_TPRIME, TPRIME_LIMITS, data_selected["data_weight"]),
}
display(pd.DataFrame({name: [np.round(edges, 6).tolist()] for name, edges in tprime_candidates.items()}))

if TPRIME_EDGE_MODE == "manual":
    if MANUAL_TPRIME_EDGES is None:
        raise ValueError("Set MANUAL_TPRIME_EDGES when TPRIME_EDGE_MODE='manual'")
    TPRIME_EDGES = np.asarray(MANUAL_TPRIME_EDGES, dtype=float)
elif TPRIME_EDGE_MODE in tprime_candidates:
    TPRIME_EDGES = tprime_candidates[TPRIME_EDGE_MODE].copy()
else:
    raise ValueError(f"Unknown TPRIME_EDGE_MODE: {TPRIME_EDGE_MODE}")

if len(TPRIME_EDGES) != N_TPRIME + 1 or np.any(np.diff(TPRIME_EDGES) <= 0.0):
    raise ValueError("TPRIME_EDGES must contain N_TPRIME + 1 strictly increasing edges")
if not np.allclose([TPRIME_EDGES[0], TPRIME_EDGES[-1]], TPRIME_LIMITS):
    raise ValueError("TPRIME_EDGES endpoints must equal TPRIME_LIMITS")

TPRIME_CENTERS = 0.5 * (TPRIME_EDGES[:-1] + TPRIME_EDGES[1:])
PHI_CENTERS = 0.5 * (PHI_EDGES[:-1] + PHI_EDGES[1:])
print(f"Chosen data-only t' mode: {TPRIME_EDGE_MODE}")
print("t' edges [GeV^2]:", np.round(TPRIME_EDGES, 6))
print("t' centers [GeV^2]:", np.round(TPRIME_CENTERS, 6))
print("phi centers [deg]:", np.round(np.degrees(PHI_CENTERS), 1))

,uniform,count_quantile,yield_quantile
0,"[-0.7, -0.56, -0.42, -0.28, -0.14, 0.0]","[-0.7, -0.319544, -0.221239, -0.153472, -0.086719, 0.0]","[-0.7, -0.319537, -0.218896, -0.150272, -0.082709, 0.0]"


Chosen data-only t' mode: count_quantile
t' edges [GeV^2]: [-0.7      -0.319544 -0.221239 -0.153472 -0.086719  0.      ]
t' centers [GeV^2]: [-0.509772 -0.270392 -0.187355 -0.120095 -0.043359]
phi centers [deg]: [ 12.  36.  60.  84. 108. 132. 156. 180. 204. 228. 252. 276. 300. 324.
 348.]


In [15]:
# Create the linked dashboard once. Rerun this cell to reset its controls to the configuration cell.
import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.xsec_extract.xsec_bin_dashboard import BinDashboard

def receive_dashboard_selection(view):
    global data_selected, selected_frames, TPRIME_LIMITS, Q2_LIMITS, XB_LIMITS
    global DIAMOND_XB_Q2_VERTICES, N_TPRIME, N_Q2, N_XB_PER_Q2, N_PHI
    global TPRIME_EDGE_MODE, Q2_EDGE_MODE, XB_EDGE_MODE, PHI_EDGES
    global TPRIME_EDGES, TPRIME_CENTERS, PHI_CENTERS
    global MANUAL_TPRIME_EDGES, MANUAL_Q2_EDGES, XB_EDGES_BY_Q2
    selected_frames = view.selected_frames
    data_selected = pd.concat(selected_frames, ignore_index=True)
    TPRIME_LIMITS, Q2_LIMITS, XB_LIMITS = view.t_limits.value, view.q_limits.value, view.x_limits.value
    N_TPRIME, N_Q2, N_XB_PER_Q2, N_PHI = view.n_t.value, view.n_q.value, view.n_x.value, view.n_phi.value
    TPRIME_EDGE_MODE, Q2_EDGE_MODE, XB_EDGE_MODE = view.t_mode.value, view.q_mode.value, view.x_mode.value
    MANUAL_TPRIME_EDGES = view.manual_edges.get("tprime")
    MANUAL_Q2_EDGES = view.manual_edges.get("Q2")
    XB_EDGES_BY_Q2 = view.manual_edges.get("xB") if XB_EDGE_MODE == "manual" else None
    if not view.valid:
        TPRIME_EDGES = TPRIME_CENTERS = PHI_EDGES = PHI_CENTERS = np.array([])
        return
    polygon = view._polygon()
    DIAMOND_XB_Q2_VERTICES = None if polygon is None else polygon.tolist()
    TPRIME_EDGES = view.edges["tprime"].copy()
    PHI_EDGES = view.edges["phi"].copy()
    TPRIME_CENTERS = (TPRIME_EDGES[:-1] + TPRIME_EDGES[1:]) / 2
    PHI_CENTERS = (PHI_EDGES[:-1] + PHI_EDGES[1:]) / 2

dashboard = BinDashboard(
    settings=KINEMATICS, coverage_frames=coverage_frames, proton_mass=PROTON_MASS_GEV,
    limits={"tprime": TPRIME_LIMITS, "Q2": Q2_LIMITS, "xB": XB_LIMITS},
    modes={"tprime": TPRIME_EDGE_MODE, "Q2": Q2_EDGE_MODE, "xB": XB_EDGE_MODE},
    counts={"tprime": N_TPRIME, "Q2": N_Q2, "xB": N_XB_PER_Q2, "phi": N_PHI},
    vertices=DIAMOND_XB_Q2_VERTICES,
    manual_edges={"tprime": MANUAL_TPRIME_EDGES, "Q2": MANUAL_Q2_EDGES, "xB": XB_EDGES_BY_Q2},
    on_change=receive_dashboard_selection,
    overlap_bins=OVERLAP_BINS, w_q2_bins=W_Q2_MAP_BINS,
    min_events_per_cell=MIN_EVENTS_PER_CELL,
    max_scatter=MAX_SCATTER_PER_SETTING,
)
dashboard.show()
display(pd.DataFrame(dashboard.projection_checks))


,setting,max_abs_W_delta_GeV
0,KinC_x36_4,4.440892e-16
1,KinC_x36_5_407,4.440892e-16


## 4. Review edges and diamond for the xsec configs

After editing the interactive dashboard, run the cell below. It reads the exact current edges, shows per-setting occupancy, and prints a JSON block for `xsec_config/xsec_config_*.json`. Copy its `phi_bins`, `tprime_bin_edges`, `q2_bin_edges`, `xb_bin_edges_by_q2`, and `diamond_xb_q2_vertices` fields into each setting you will extract. Each xB row belongs to the corresponding interval in `q2_bin_edges`. The xsec config generator requires all xB rows to have the same number of edges and matching outer endpoints. The diamond corners are `[xB, Q2]` pairs; the W/Q2 boundary is derived from them. `null` means no diamond cut. Keep `t_bin_edges` as a separate setting.

To freeze automatic Q2/xB choices, paste the printed Q2 array and xB array-of-arrays into the dashboard **Manual edges** fields and press **Apply manual arrays**. Rerun this cell and check for `Final manual edges`. The dashboard **Copy configuration for a future run** panel preserves those choices for the notebook configuration cell. `WRITE_EXPORT` controls the separate scientific JSON export; it is not needed to copy edges into the xsec configs.

In [16]:
# Use the exact arrays shown by the dashboard; do not derive another binning.
dashboard.require_ready()
edge_review_revision = dashboard.revision
q2_edges_final = dashboard.edges["Q2"].copy()
xb_edges_by_q2_final = [e.copy() for e in dashboard.edges["xB_by_Q2"]]
Q2_XB_FINALIZED = dashboard.q_mode.value == "manual" and dashboard.x_mode.value == "manual"
print("Q2 edges:", q2_edges_final)
for iq, edges in enumerate(xb_edges_by_q2_final):
    print(f"xB edges in Q2 bin {iq}: {edges}")
print("Final manual edges:" if Q2_XB_FINALIZED else "Preview edges; scientific JSON export requires manual Q2/xB")
config_edges = {
    "phi_bins": N_PHI,
    "tprime_bin_edges": TPRIME_EDGES.tolist(),
    "q2_bin_edges": q2_edges_final.tolist(),
    "xb_bin_edges_by_q2": [edges.tolist() for edges in xb_edges_by_q2_final],
    "diamond_xb_q2_vertices": DIAMOND_XB_Q2_VERTICES,
}
if len({len(edges) for edges in xb_edges_by_q2_final}) != 1:
    print("WARNING: xsec_config requires the same number of xB edges in every Q2 bin")
print("Copy these fields into each selected xsec_config JSON:")
print(json.dumps(config_edges, indent=2))

coverage_rows = []
for setting, frame in zip(KINEMATICS, selected_frames):
    for iq, xb_edges in enumerate(xb_edges_by_q2_final):
        qmask = (frame["Q2"] >= q2_edges_final[iq]) & (frame["Q2"] <= q2_edges_final[iq + 1] if iq == len(q2_edges_final) - 2 else frame["Q2"] < q2_edges_final[iq + 1])
        for ix in range(len(xb_edges) - 1):
            xmask = (frame["xB"] >= xb_edges[ix]) & (frame["xB"] <= xb_edges[ix + 1] if ix == len(xb_edges) - 2 else frame["xB"] < xb_edges[ix + 1])
            coverage_rows.append({"setting": setting, "q2_bin": iq, "xb_bin": ix, "events": int((qmask & xmask).sum())})
display(pd.DataFrame(coverage_rows))

Q2 edges: [3.         3.90624907 5.        ]
xB edges in Q2 bin 0: [0.25       0.32366883 0.455     ]
xB edges in Q2 bin 1: [0.25       0.38483385 0.455     ]
Preview edges; scientific JSON export requires manual Q2/xB
Copy these fields into each selected xsec_config JSON:
{
  "phi_bins": 15,
  "tprime_bin_edges": [
    -0.7,
    -0.31954380358349815,
    -0.2212392283189899,
    -0.15347165586141714,
    -0.08671872370869665,
    0.0
  ],
  "q2_bin_edges": [
    3.0,
    3.906249068514029,
    5.0
  ],
  "xb_bin_edges_by_q2": [
    [
      0.25,
      0.32366883416346043,
      0.455
    ],
    [
      0.25,
      0.3848338493330195,
      0.455
    ]
  ],
  "diamond_xb_q2_vertices": null
}


,setting,q2_bin,xb_bin,events
0,KinC_x36_4,0,0,2050
1,KinC_x36_4,0,1,2676
2,KinC_x36_4,1,0,2757
3,KinC_x36_4,1,1,2345
4,KinC_x36_5_407,0,0,3331
5,KinC_x36_5_407,0,1,2706
6,KinC_x36_5_407,1,0,2625
7,KinC_x36_5_407,1,1,3037


## 5. SIMC-derived representative $Q^2$ and $x_B$ after binning is frozen

This is the first section that opens the simulation for `REFERENCE_KINEMATIC`. It cannot alter any edge. Reconstructed variables decide which accepted exclusive event belongs to a frozen bin. The requested representative values use `full_weight` and generated $Q_i^2$ plus $x_{B,i}=Q_i^2/(W_i^2-M_p^2+Q_i^2)$. Reconstructed weighted means are retained only as diagnostics. All $\phi$ bins are combined.

These are accepted-SIMC, model-weighted kinematic means; they are not efficiencies or acceptance corrections.

In [17]:
dashboard.require_ready(edge_review_revision)
SIM_BRANCHES = [
    "Q2", "xB", "t", "tmin", "phi", "mmiss", "full_weight",
    "is_exclusive", "Q2i", "Wi",
]
if SIM_FILE is None or not SIM_FILE.is_file():
    sim_selected = pd.DataFrame(columns=SIM_BRANCHES + ["tprime", "phi_wrapped", "xBi"])
    print(f"No SIMC input configured for {REFERENCE_KINEMATIC}; representative coordinates unavailable")
else:
    sim_root, sim_tree = open_checked_tree(SIM_FILE, SIM_TREE, SIM_BRANCHES)
    sim = tree_frame(sim_tree, SIM_BRANCHES)
    sim_root.close()
    sim["tprime"] = sim["t"] - sim["tmin"]
    sim["phi_wrapped"] = np.mod(sim["phi"], 2.0 * np.pi)
    xb_i_denominator = sim["Wi"] ** 2 - PROTON_MASS_GEV ** 2 + sim["Q2i"]
    sim["xBi"] = np.where(xb_i_denominator > 0.0, sim["Q2i"] / xb_i_denominator, np.nan)
    sim_finite = np.logical_and.reduce([np.isfinite(sim[name]) for name in ["Q2", "xB", "tprime", "phi_wrapped", "mmiss", "full_weight"]])
    sim_selected_mask = (
        sim_finite & (sim["is_exclusive"] != 0) & (sim["full_weight"] >= 0.0)
        & sim["mmiss"].between(*MMISS_LIMITS, inclusive="both")
        & sim["tprime"].between(TPRIME_EDGES[0], TPRIME_EDGES[-1], inclusive="both")
        & sim["Q2"].between(q2_edges_final[0], q2_edges_final[-1], inclusive="both")
        & sim["xB"].between(*XB_LIMITS, inclusive="both") & diamond_mask(sim)
    )
    sim_selected = sim.loc[sim_selected_mask].copy()

def weighted_mean(values, weights):
    values, weights = np.asarray(values, dtype=float), np.asarray(weights, dtype=float)
    keep = np.isfinite(values) & np.isfinite(weights) & (weights > 0.0)
    return np.average(values[keep], weights=weights[keep]) if np.any(keep) else np.nan

def in_bin(values, edges, index):
    upper = values <= edges[index + 1] if index == len(edges) - 2 else values < edges[index + 1]
    return (values >= edges[index]) & upper

average_rows = []
for it in range(N_TPRIME):
    tmask_sim = in_bin(sim_selected["tprime"], TPRIME_EDGES, it)
    reference_data = data_selected.loc[data_selected["setting"] == REFERENCE_KINEMATIC]
    tmask_data = in_bin(reference_data["tprime"], TPRIME_EDGES, it)
    for iq in range(len(q2_edges_final) - 1):
        qmask_sim = in_bin(sim_selected["Q2"], q2_edges_final, iq)
        qmask_data = in_bin(reference_data["Q2"], q2_edges_final, iq)
        xb_edges = xb_edges_by_q2_final[iq]
        for ix in range(len(xb_edges) - 1):
            mask_sim = tmask_sim & qmask_sim & in_bin(sim_selected["xB"], xb_edges, ix)
            mask_data = tmask_data & qmask_data & in_bin(reference_data["xB"], xb_edges, ix)
            accepted = sim_selected.loc[mask_sim]
            weights = accepted["full_weight"].to_numpy()
            sumw = weights.sum()
            sumw2 = np.square(weights).sum()
            average_rows.append({
                "tprime_bin": it, "q2_bin": iq, "xb_bin": ix,
                "tprime_center": TPRIME_CENTERS[it],
                "data_events_phi_integrated": int(mask_data.sum()),
                "accepted_sim_events_phi_integrated": int(mask_sim.sum()),
                "accepted_sim_neff": (sumw * sumw / sumw2) if sumw2 > 0.0 else 0.0,
                "Q2_acceptance_weighted": weighted_mean(accepted["Q2i"], weights),
                "xB_acceptance_weighted": weighted_mean(accepted["xBi"], weights),
                "Q2_reco_weighted_diagnostic": weighted_mean(accepted["Q2"], weights),
                "xB_reco_weighted_diagnostic": weighted_mean(accepted["xB"], weights),
            })
acceptance_weighted_table = pd.DataFrame(average_rows)
print(f"Accepted exclusive SIMC entries in configured window: {len(sim_selected):,}")
display(acceptance_weighted_table.round(6))
representatives_revision = dashboard.revision


Accepted exclusive SIMC entries in configured window: 141,931


,tprime_bin,q2_bin,xb_bin,tprime_center,data_events_phi_integrated,accepted_sim_events_phi_integrated,accepted_sim_neff,Q2_acceptance_weighted,xB_acceptance_weighted,Q2_reco_weighted_diagnostic,xB_reco_weighted_diagnostic
0,0,0,0,-0.509772,607,8278,7156.539551,3.469296,0.302065,3.455854,0.298671
1,0,0,1,-0.509772,420,4676,4185.114746,3.771206,0.346875,3.764807,0.344098
2,0,1,0,-0.509772,511,7952,7151.066406,4.097598,0.367208,4.092800,0.364304
3,0,1,1,-0.509772,820,11325,10607.996094,4.447516,0.419444,4.446971,0.416946
4,1,0,0,-0.270392,608,6336,5071.265625,3.459797,0.303253,3.448995,0.300223
5,1,0,1,-0.270392,522,4687,4151.718750,3.755569,0.347442,3.749538,0.344645
6,1,1,0,-0.270392,490,6689,5939.467285,4.096252,0.367816,4.091133,0.364883
7,1,1,1,-0.270392,603,8053,7666.890625,4.430357,0.416913,4.427640,0.414128
8,2,0,0,-0.187355,673,5939,4750.242188,3.461011,0.303908,3.447903,0.300773
9,2,0,1,-0.187355,562,5246,4644.927246,3.750317,0.347230,3.744735,0.344525


## 6. Review coordinates and optionally export

The $t'$ and $\phi$ coordinates below are geometric centers by construction. The $Q^2$/$x_B$ table above is integrated over all $\phi$ bins. Export is permitted only after explicit data-chosen $Q^2$ and $x_B$ edges have been entered.

In [18]:
dashboard.require_ready(edge_review_revision)
dashboard.require_ready(representatives_revision)
coordinate_table = pd.DataFrame({
    "phi_bin": np.arange(N_PHI),
    "phi_lo_rad": PHI_EDGES[:-1],
    "phi_hi_rad": PHI_EDGES[1:],
    "phi_center_rad": PHI_CENTERS,
    "phi_center_deg": np.degrees(PHI_CENTERS),
})
display(Markdown("**Fixed phi coordinates**"))
display(coordinate_table.round(6))
display(Markdown("**Fixed t-prime coordinates**"))
display(pd.DataFrame({
    "tprime_bin": np.arange(N_TPRIME),
    "tprime_lo": TPRIME_EDGES[:-1],
    "tprime_hi": TPRIME_EDGES[1:],
    "tprime_center": TPRIME_CENTERS,
}).round(6))

def json_records(frame):
    cleaned = frame.astype(object).where(pd.notnull(frame), None)
    return cleaned.to_dict(orient="records")

bin_contract = {
    "schema_version": 1,
    "kinematic": REFERENCE_KINEMATIC,
    "settings_compared": KINEMATICS,
    "data_files": {key: str(DATA_FILES[key]) for key in KINEMATICS},
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "bin selection and representative coordinates only; no cross sections",
    "data_file": str(DATA_FILES[REFERENCE_KINEMATIC]),
    "data_tree": DATA_TREE,
    "sim_file": str(SIM_FILE) if SIM_FILE is not None else None,
    "sim_tree": SIM_TREE,
    "edge_provenance": "data_only",
    "tprime_definition": "t - tmin",
    "phi_definition": "phi modulo 2*pi",
    "selection": {
        "mmiss_gev": list(MMISS_LIMITS), "tprime_gev2": list(TPRIME_LIMITS),
        "q2_gev2": list(Q2_LIMITS), "xb": list(XB_LIMITS),
        "diamond_xb_q2_vertices": DIAMOND_XB_Q2_VERTICES,
    },
    "tprime_edge_mode": TPRIME_EDGE_MODE,
    "tprime_edges_gev2": TPRIME_EDGES.tolist(),
    "tprime_centers_gev2": TPRIME_CENTERS.tolist(),
    "phi_edges_rad": PHI_EDGES.tolist(),
    "phi_centers_rad": PHI_CENTERS.tolist(),
    "q2_edges_gev2": q2_edges_final.tolist(),
    "xb_edges_by_q2": [edges.tolist() for edges in xb_edges_by_q2_final],
    "q2_xb_edges_finalized": bool(Q2_XB_FINALIZED),
    "representative_q2_xb": {
        "source": "accepted generated-exclusive SIMC after reconstructed-bin selection",
        "weight": "full_weight",
        "phi_integration": f"all {N_PHI} phi bins together",
        "q2_variable": "Q2i",
        "xb_variable": "Q2i / (Wi^2 - Mp^2 + Q2i)",
        "rows": json_records(acceptance_weighted_table),
    },
}

print(json.dumps({k: v for k, v in bin_contract.items() if k != "representative_q2_xb"}, indent=2))
if WRITE_EXPORT:
    if len(KINEMATICS) != 1 or SIM_FILE is None or not SIM_FILE.is_file():
        raise RuntimeError("Export requires one setting and its configured SIMC input")
    if not Q2_XB_FINALIZED:
        raise RuntimeError("Refusing export: set data-chosen Q2_EDGES and XB_EDGES_BY_Q2 first")
    EXPORT_PATH.write_text(json.dumps(bin_contract, indent=2) + "\n", encoding="utf-8")
    print(f"Wrote {EXPORT_PATH}")
else:
    print("Preview only. Set WRITE_EXPORT=True after every edge is final.")

**Fixed phi coordinates**

,phi_bin,phi_lo_rad,phi_hi_rad,phi_center_rad,phi_center_deg
0,0,0.000000,0.418879,0.209440,12.0
1,1,0.418879,0.837758,0.628319,36.0
2,2,0.837758,1.256637,1.047198,60.0
3,3,1.256637,1.675516,1.466077,84.0
4,4,1.675516,2.094395,1.884956,108.0
5,5,2.094395,2.513274,2.303835,132.0
6,6,2.513274,2.932153,2.722714,156.0
7,7,2.932153,3.351032,3.141593,180.0
8,8,3.351032,3.769911,3.560472,204.0
9,9,3.769911,4.188790,3.979351,228.0


**Fixed t-prime coordinates**

,tprime_bin,tprime_lo,tprime_hi,tprime_center
0,0,-0.700000,-0.319544,-0.509772
1,1,-0.319544,-0.221239,-0.270392
2,2,-0.221239,-0.153472,-0.187355
3,3,-0.153472,-0.086719,-0.120095
4,4,-0.086719,0.000000,-0.043359


{
  "schema_version": 1,
  "kinematic": "KinC_x36_5_407",
  "settings_compared": [
    "KinC_x36_4",
    "KinC_x36_5_407"
  ],
  "data_files": {
    "KinC_x36_4": "/w/hallc-scshelf2102/nps/singhav/nps_analysis/pi0_analysis/root_analysis_env_main/output/KinC_x36_4/root/combined_branches_LH2.root",
    "KinC_x36_5_407": "/w/hallc-scshelf2102/nps/singhav/nps_analysis/pi0_analysis/root_analysis_env_main/output/KinC_x36_5_407/KinC_x36_5/root/combined_branches_LH2.root"
  },
  "created_utc": "2026-10-05T21:34:52.750119+00:00",
  "purpose": "bin selection and representative coordinates only; no cross sections",
  "data_file": "/w/hallc-scshelf2102/nps/singhav/nps_analysis/pi0_analysis/root_analysis_env_main/output/KinC_x36_5_407/KinC_x36_5/root/combined_branches_LH2.root",
  "data_tree": "physics",
  "sim_file": "/volatile/hallc/nps/singhav/nps_smearing/smear_x36_5_407/smearing_output/KinC_x36_5_407/root/simc_pi0_analysis_output_smeared.root",
  "sim_tree": "simulation",
  "edge_provenance": 